## Лабораторная работа № 1 
## Выполнение разведочного анализа больших данных с использованием фреймворка Apache Spark

### Часть 1

В данной части работы рассмотрены:
* загрузка данных из HDFS;
* базовые преобразования данных;
* загрузка преобразованных данных в таблицу `Apache Airflow`.

Подключим необходимые библиотеки.

In [1]:
import os
from pyspark.sql import SparkSession, DataFrame
from pyspark import SparkConf
from pyspark.sql.functions import (
    regexp_replace,
    regexp_extract_all,
    regexp_extract,
    col,
    lit,
    when,
    to_date,
    from_unixtime,
    split,
    trim,
    year
)

Сформируем объект конфигурации для `Apache Spark`, указав необходимые параметры.

In [2]:
def create_spark_configuration() -> SparkConf:
    """
    Создает и конфигурирует экземпляр SparkConf для приложения Spark.

    Returns:
        SparkConf: Настроенный экземпляр SparkConf.
    """
    # Получаем имя пользователя
    # Получаем имя пользователя
    user_name = os.getenv("USER")
    
    conf = SparkConf()
    conf.setAppName("lab 1")
    conf.setMaster("local[*]")
    # conf.set("spark.submit.deployMode", "client")
    conf.set("spark.executor.memory", "12g")
    conf.set("spark.executor.cores", "6")
    conf.set("spark.executor.instances", "1")
    conf.set("spark.driver.memory", "4g")
    conf.set("spark.driver.cores", "1")

    return conf

Создаём сам объект конфигурации.

In [3]:
conf = create_spark_configuration()

Создаём и выводим на экран сессию `Apache Spark`. В процессе создания сессии происходит подключение к кластеру `Apache Hadoop`, что может занять некоторое время.

In [4]:
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark

Для исследования будем использовать датасет `"Big Sales Data"`, расположенный на платформе `Kaggle` по адресу https://www.kaggle.com/datasets/pigment/big-sales-data/data.

Указываем путь.

In [5]:
path1 = 'data/Books_rating.csv'
path2 = 'data/books_data.csv'

Заполняем датафрейм данными из файла.

In [6]:
df1 = (spark.read.format("csv")
      .option("header", "true")
      .load(path1)
)

df2 = (spark.read.format("csv")
      .option("header", "true")
      .load(path2)
)

df = df1.join(df2, "Title", "inner")

Выводим фрагмент датафрейма на экран.

In [7]:
df.show()

+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|               Title|        Id|               Price|             User_id|         profileName|  review/helpfulness|        review/score|         review/time|      review/summary|         review/text|         description|             authors|               image|         previewLink|           publisher|       publishedDate|            infoLink|          categories|        ratingsCount|
+--------------------+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------

Очевидно, что в целях сохранения ясности изложения и сокращения расчетного времени имеет смысл рассматривать не все столбцы датасета. Оставим следующие колонки, удалив остальные:

### Анализ отзывов и рейтингов
| Название столбца | Расшифровка |
| -------- | ------ |
| id |	Уникальный идентификатор книги (ISBN/ASIN)
| title	| Название книги
| review_score	| Оценка книги пользователем (1.0-5.0)
| review_text	| Текст отзыва
| review_summary	| Заголовок отзыва
| review_helpfulness	| Полезность отзыва (в формате "X/Y")
| review_time	| Временная метка отзыва

### Анализ книг и метаданных
| Название столбца | Расшифровка |
| -------- | ------ |
| authors	| Авторы книги
| publisher	| Издательство
| published_date	| Дата публикации книги
| categories	| Категории/жанры книги

In [8]:
df = df.select(
    "Id", "Title", "authors", "publisher", "publishedDate", "categories",
    "review/score", "review/text", "review/summary", "review/helpfulness",
    "review/time", "User_id", "profileName"
)

In [9]:
df.show()

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|        Id|               Title|             authors|           publisher|       publishedDate|          categories|        review/score|         review/text|      review/summary|  review/helpfulness|         review/time|             User_id|         profileName|
+----------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+--------------------+
|B0007H2HAM|"""Always ready!"...|&dq=%22Always+rea...|                1943|http://books.goog...|                NULL| it's the narrati...| 1941) which was ...| Bell had already...|"It's hard to bel...| Ken

Выведем на экран метаданные датасета.

In [10]:
df.printSchema()

root
 |-- Id: string (nullable = true)
 |-- Title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- publishedDate: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- review/score: string (nullable = true)
 |-- review/text: string (nullable = true)
 |-- review/summary: string (nullable = true)
 |-- review/helpfulness: string (nullable = true)
 |-- review/time: string (nullable = true)
 |-- User_id: string (nullable = true)
 |-- profileName: string (nullable = true)



Видно, что все столбцы датасета содержат строковый тип данных, что не соответствует ожиданиям. Выполним преобразования типов данных некоторых столбцов.

In [11]:
def transform_dataframe(data: DataFrame) -> DataFrame:
    """
    Преобразует столбцы DataFrame в указанные типы данных и
    выполняет необходимые преобразования.

    Args:
        data (DataFrame): Исходный DataFrame.

    Returns:
        DataFrame: Преобразованный DataFrame.
    """
    data = data \
        .withColumnRenamed("Id", "id") \
        .withColumnRenamed("Title", "title") \
        .withColumnRenamed("review/time", "review_time") \
        .withColumnRenamed("review/helpfulness", "review_helpfulness") \
        .withColumnRenamed("review/summary", "review_summary") \
        .withColumnRenamed("review/text", "review_text") \
        .withColumnRenamed("review/score", "review_score") \
        .withColumnRenamed("publishedDate", "published_date") \
        .withColumnRenamed("User_id", "user_id") \
        .withColumnRenamed("profileName", "profile_name")
    
    # Преобразуем столбцы в соответствующие типы данных
    data = data \
        .withColumn("id", col("id").cast("string")) \
        .withColumn("review_score", col("review_score").cast("double")) \
        .withColumn("review_time", col("review_time").cast("integer")) \
        .withColumn("user_id", col("user_id").cast("string"))
    
    # Извлекаем числовую часть из review_helpfulness
    data = data.withColumn(
        "helpfulness_numerator", 
        regexp_extract(col("review_helpfulness"), r"(\d+)/\d+", 1).cast("integer")
    ).withColumn(
        "helpfulness_denominator", 
        regexp_extract(col("review_helpfulness"), r"\d+/(\d+)", 1).cast("integer")
    ).withColumn(
        "helpfulness_ratio", 
        when(col("helpfulness_denominator") > 0, 
             col("helpfulness_numerator") / col("helpfulness_denominator"))
        .otherwise(0.0)
    )
    
    # Преобразуем publishedDate в дату (обрабатываем разные форматы)
    data = data.withColumn(
    "published_date_clean",
    when(col("published_date").rlike(r"^\\d{4}$"),  # только год
         to_date(lit("01-01-") + col("published_date"), "dd-MM-yyyy"))
    .when(col("published_date").rlike(r"^\\d{4}-\\d{2}-\\d{2}$"),  # YYYY-MM-DD
         to_date(col("published_date"), "yyyy-MM-dd"))
    .otherwise(None)
)

    # Преобразуем authors и categories в массивы (если они в строковом формате списка)
    data = data.withColumn(
        "authors_array",
        when(col("authors").rlike(r"^\[.*\]$"),
             regexp_extract_all(col("authors"), lit(r"'([^']*)'"), 1))
        .otherwise(split(col("authors"), ","))
    )
    
    data = data.withColumn(
        "categories_array",
        when(col("categories").rlike(r"^\[.*\]$"),
             regexp_extract_all(col("categories"), lit(r"'([^']*)'"), 1))
        .otherwise(split(col("categories"), ","))
    )
    
    # Очистка текстовых полей от лишних пробелов и специальных символов
    text_columns = ["title", "publisher", "review_summary", "review_text", "profile_name"]
    
    for col_name in text_columns:
        data = data.withColumn(
            col_name,
            when(col(col_name).isNotNull(), 
                 trim(regexp_replace(col(col_name), r"\s+", " ")))
            .otherwise(col(col_name))
        )
    
    # Создаем год публикации для анализа
    data = data.withColumn(
        "published_year",
        year(col("published_date_clean"))
    )
    
    return data


In [12]:
df = transform_dataframe(df)

In [13]:
df.show()

+----------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+-----------+--------------------+--------------------+---------------------+-----------------------+------------------+--------------------+--------------------+--------------------+--------------+
|        id|               title|             authors|           publisher|      published_date|          categories|review_score|         review_text|      review_summary|  review_helpfulness|review_time|             user_id|        profile_name|helpfulness_numerator|helpfulness_denominator| helpfulness_ratio|published_date_clean|       authors_array|    categories_array|published_year|
+----------+--------------------+--------------------+--------------------+--------------------+--------------------+------------+--------------------+--------------------+--------------------+-----------+-------------

In [14]:
df.printSchema()

root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- published_date: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- review_score: double (nullable = true)
 |-- review_text: string (nullable = true)
 |-- review_summary: string (nullable = true)
 |-- review_helpfulness: string (nullable = true)
 |-- review_time: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- profile_name: string (nullable = true)
 |-- helpfulness_numerator: integer (nullable = true)
 |-- helpfulness_denominator: integer (nullable = true)
 |-- helpfulness_ratio: double (nullable = true)
 |-- published_date_clean: date (nullable = true)
 |-- authors_array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories_array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- published_year: integer (nullable = true)



# Сохранение DataFrame в формате Parquet

### Запись данных из Parquet

In [15]:
df.write \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet("sobd_lab1_table_parquet")

print("Данные сохранены в sobd_lab1_table_parquet")

Данные сохранены в sobd_lab1_table_parquet


### Чтение данных из Parquet

In [16]:
df_parquet = spark.read.parquet("sobd_lab1_table_parquet")

print("Данные успешно загружены из Parquet")
print(f"Количество строк: {df_parquet.count()}")
print(f"Количество колонок: {len(df_parquet.columns)}")

# Показать схему данных
print("Схема данных:")
df_parquet.printSchema()

# Показать первые несколько строк
print("Первые 10 строк:")
df_parquet.show(10)

Данные успешно загружены из Parquet
Количество строк: 2999829
Количество колонок: 20
Схема данных:
root
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- published_date: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- review_score: double (nullable = true)
 |-- review_text: string (nullable = true)
 |-- review_summary: string (nullable = true)
 |-- review_helpfulness: string (nullable = true)
 |-- review_time: integer (nullable = true)
 |-- user_id: string (nullable = true)
 |-- profile_name: string (nullable = true)
 |-- helpfulness_numerator: integer (nullable = true)
 |-- helpfulness_denominator: integer (nullable = true)
 |-- helpfulness_ratio: double (nullable = true)
 |-- published_date_clean: date (nullable = true)
 |-- authors_array: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories_array: array (nullable = true)

После успешной записи таблицы останавливаем сессию `Apache Spark`.

In [17]:
spark.stop()